# 07 · Aprovechar lo que una red ya ha aprendido de otras fotos

**Pregunta:** ¿podemos reconocer nuestras prendas aprovechando una red que ya aprendió a mirar imágenes, en vez de enseñárselo todo desde cero?

Usamos **DINO con ResNet50**, un modelo visual preentrenado mediante autosupervisión. Recibe **fotos**, no descripciones de texto. Conservamos los productos y las particiones del 02 vigente y trabajamos con **64 × 64 píxeles**, como las redes del 04, 05 y 06.

> **Estado: implementación preparada, pendiente de ejecución y validación por la estudiante.** No hay resultados que permitan afirmar que este modelo mejora los anteriores. Las operaciones costosas están apagadas por defecto. Ejecutar todo con las opciones en `False` carga resultados compatibles si existen y explica lo que falta si todavía no existen.

Este cuaderno desarrolla la **transferencia con la red fija**. El ajuste de la parte que mira las fotos se explica, pero queda pendiente de una decisión posterior. No se activa por sorpresa.


## 1. Qué vamos a entrenar, explicado con dos partes

Imaginemos que el modelo tiene dos partes:

| Parte | Qué hace | Qué cambiamos aquí |
|---|---|---|
| **La que mira la foto** | Reconoce patrones y los resume en una lista de números. | Nada: aprovechamos lo que ya aprendió. |
| **La que decide la prenda** | Usa ese resumen para elegir entre camiseta, pantalón, abrigo… | Esta parte sí aprende con nuestras etiquetas. |

La primera parte se llama **encoder** o **extractor de características**. Su lista de números se llama **vector de características** o **embedding**. En este caso contiene 2.048 números: son un resumen aprendido de la imagen, no palabras ni probabilidades de prendas.

La segunda parte es un **clasificador lineal**: combina esos números para dar una puntuación a cada clase. “Lineal” significa que esta última parte es una sola capa, sin añadir otra red profunda.

**Transferencia** significa aprovechar lo aprendido con otros datos para nuestra tarea. **Congelar la red** significa dejar fija la parte que mira las fotos. Además de impedir cambios en sus pesos, usamos el modo de evaluación: tampoco actualizamos sus estadísticas internas de BatchNorm.

**¿Y ajustar la red, o fine-tuning?** Sería permitir que también cambiase parte de la que mira las fotos para adaptarse a nuestra ropa. Puede ayudar, pero necesita más cálculo y puede memorizar demasiado los ejemplos de entrenamiento. Primero mediremos el resultado y el coste de la opción sencilla. No hemos decidido qué capas ajustar ni cuánto tiempo: ese experimento no está implementado ni se ejecuta aquí.


### 1.1. Qué significan DINO y ResNet50

**ResNet50 es la red que procesa las imágenes. DINO es el método con el que se entrenó previamente.** DINO también se ha utilizado con Transformers, pero el checkpoint elegido aquí usa una CNN ResNet50; no sustituye al experimento de Transformer del 08.

Durante aquel entrenamiento se mostraban versiones distintas de una misma foto. Una red “alumna” aprendía a producir respuestas coherentes con otra red “profesora”. Esta profesora se actualizaba mediante una media de los pesos de la alumna, no mediante etiquetas humanas de las imágenes. Es una explicación inicial de la **autodestilación sin etiquetas** de DINO.

Los autores ya realizaron ese trabajo con ImageNet-1k. **Nosotros reutilizamos sus pesos, sin repetir su preentrenamiento.** El entrenamiento de nuestro clasificador sí es supervisado: usa las etiquetas de las prendas de train. Por tanto, hay dos momentos distintos: preentrenamiento autosupervisado externo y adaptación supervisada local.

Esto cubre la aplicación de un modelo preentrenado mediante autosupervisión y permite estudiar transferencia. No demuestra que hayamos preentrenado un modelo fundacional propio. Tampoco significa que ResNet50 sea un gran modelo multimodal.

Fuentes: [artículo de DINO](https://arxiv.org/abs/2104.14294), [repositorio de los autores](https://github.com/facebookresearch/dino) y [carga oficial del checkpoint ResNet50](https://github.com/facebookresearch/dino/blob/main/hubconf.py).


## 2. Por qué usamos 64 × 64 y qué comparación permite

**Sí, tiene sentido para nuestro objetivo:** estudiar distintas formas de clasificar las mismas prendas con imágenes del mismo tamaño. Es una decisión acordada antes de ver los resultados del 07. Reducir una foto a 64 × 64 no significa elegir otras prendas ni quitar ejemplos.

| Cuaderno | Qué recibe el método | Tamaño utilizado |
|---|---|---|
| 02 · Preparación | Guarda una copia de trabajo de cada foto completa, con relleno blanco para conservar la proporción. | 224 × 224 |
| 03 · Referencias sencillas | Resume el color del contenido de la foto; no es una CNN de píxeles. | Copia de 32 × 32 para calcular los histogramas |
| 04 · Dense | Lee los píxeles como una lista. | 64 × 64 |
| 05 · CNN | Aprende formas locales y las combina. | 64 × 64 |
| 06 · RNN, LSTM y GRU | Lee la imagen como una secuencia de columnas; también estudia otras representaciones. | 64 × 64 |
| **07 · DINO + clasificador** | Resume la foto con ResNet50 fija y aprende la última capa. | **64 × 64** |

**El límite de esta decisión:** la evaluación publicada por los autores de DINO usa un procedimiento a 224 × 224. Nuestro experimento no reproduce esa evaluación y puede perder detalles útiles al reducir la imagen. Un resultado bajo a 64 × 64 no demostraría que DINO es malo en general.

El ejemplo original al que nos referimos es **el código público de los autores de DINO**, no un notebook del profesor. Ese código redimensiona y recorta el centro para evaluar a 224 × 224. Aquí reducimos directamente la copia completa del 02 a 64 × 64 con el mismo método Lanczos utilizado en los arrays de 04–06. **No añadimos otro recorte que pueda cortar la prenda.** Tampoco se cambia el filtro de zooms del 02.

**Qué queda igual:** productos, clases actuales, particiones y resolución de las redes 04–07. **Qué sigue cambiando:** arquitectura, número de parámetros, aprendizaje previo y preprocesamiento. Por ejemplo, aquí respetamos la normalización de DINO; 04–06 usan los momentos de train. Los límites de épocas tampoco son iguales en todos los cuadernos: la configuración actual del 05 permite 10 y la del 06 permite 30.

Podemos comparar soluciones prácticas, siempre que sus resultados correspondan al **mismo manifiesto de datos**. No podemos atribuir toda diferencia únicamente al preentrenamiento. Para aislar ese efecto habría que comparar la misma ResNet50 con pesos aleatorios y preentrenados bajo un protocolo controlado; **ese contraste de H3 queda pendiente y no se simula con los resultados de una CNN diferente**.

Referencia del tamaño y preprocesamiento originales: [evaluación lineal de DINO](https://github.com/facebookresearch/dino/blob/main/eval_linear.py).


### 2.1. Normalización y aumentos

Primero convertimos los píxeles de 0–255 a 0–1. Después aplicamos por canal la operación `(valor − media) / desviación`, con los valores fijos usados en la evaluación de DINO:

- Media RGB: `[0.485, 0.456, 0.406]`.
- Desviación RGB: `[0.229, 0.224, 0.225]`.

Esto adapta la escala de los números a la que espera el modelo ya entrenado. **No calculamos estos valores con validation ni test**: son constantes externas del preprocesamiento del checkpoint. Tampoco los confundimos con los momentos calculados con train en otros experimentos.

En esta primera prueba no añadimos aumentos ni pesos de clase. Así estudiamos una transferencia sencilla con la distribución natural conservada por el 02. Las variantes de aumentos suaves y fuertes siguen siendo experimentos que deben identificarse por separado.


## 3. Configuración: por defecto, cargar; no volver a entrenar

Selecciona el kernel **.venv**. Las dependencias adicionales están en `requirements-transfer.txt`; el cuaderno no las instala. Si faltan, la instalación se hace una vez en ese entorno y después se reinicia el kernel:

```python
%pip install -r "../requirements-transfer.txt"
```

La ruta anterior presupone que la carpeta de trabajo es `notebooks`; si no lo es, usa la ruta completa al archivo. **No ejecutes una instalación si solo vas a ver resultados guardados y tu entorno ya funciona.**

| Opción | Si vale True | Valor habitual |
|---|---|---|
| DESCARGAR_PESOS | Descarga los pesos oficiales si faltan. Necesita internet. | False |
| MEDIR_COSTE | Analiza hasta tres lotes de fotos para medir el coste. Guarda sus vectores y la medición. | False |
| EXTRAER_FOTOS | Analiza las fotos que todavía no tienen vector guardado. Exige una medición previa. | False |
| ENTRENAR_CLASIFICADOR | Entrena solo la pequeña capa final, si faltan resultados. | False |

No hay descarga, extracción ni entrenamiento por el mero hecho de que falte un archivo. La medición no recorta el dataset: sus fotos también forman parte del experimento completo. El número final de clases se lee del 02, no se escribe a mano.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, Image as DisplayImage

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'config/transfer_v1.json').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.visualization.style import apply_style, display_es, BLUE, ORANGE
from src.training.transfer_experiment import (
    load_context, experiment_identity, saved_experiment, prepare_weights,
    feature_status, extract_features, train_heads, cost_path,
)
from src.training.transfer_results import evaluate_saved, product_bootstrap

DESCARGAR_PESOS = False
MEDIR_COSTE = False
EXTRAER_FOTOS = False
ENTRENAR_CLASIFICADOR = False

apply_style()
context = load_context(ROOT)
config = context['config']
saved = saved_experiment(context)
display(Markdown(f"**Dataset activo:** `{context['run'].name}` · "
                 f"**{len(context['names'])} clases** · **64 × 64**."))
display_es(pd.DataFrame([
    {'Partición': split, 'Imágenes': len(rows), 'Grupos': rows.group_id.nunique()}
    for split, rows in context['rows'].items()
]))
display_es(pd.DataFrame({
    'Decisión': ['Parte que mira la foto', 'Parte que aprende', 'Semillas',
                 'Máximo de épocas', 'Parada', 'Aumentos', 'Pesos de clase'],
    'Valor': ['DINO / ResNet50 fija', 'Una capa lineal', str(config['training']['seeds']),
              config['training']['max_epochs'], '5 épocas sin mejorar la pérdida de validation',
              'No', 'No'],
}))
display(Markdown('**Resultados compatibles encontrados.** No hace falta repetir el trabajo.'
                 if saved is not None else
                 '**Todavía no hay un entrenamiento completo compatible.** '
                 'Con las cuatro opciones en False, el cuaderno solo explica lo que falta.'))


### 3.1. Entender exactamente la capa que sí aprende

Cada foto se convierte en una lista de 2.048 números. Una capa `Linear(2048, número_de_clases)` produce una puntuación por clase. Sus parámetros entrenables son `(2048 + 1) × número_de_clases`: el +1 corresponde al término independiente de cada clase.

La siguiente celda **define una función**, pero no crea ni entrena el modelo. El entrenamiento de la sección 6 utiliza esta misma arquitectura y la misma pérdida. El código completo se puede consultar en `src/training/transfer_experiment.py`.


In [ ]:
def ejemplo_del_clasificador(numero_de_clases):
    from torch import nn
    clasificador = nn.Linear(2048, numero_de_clases)
    perdida = nn.CrossEntropyLoss()
    return clasificador, perdida

display(Markdown(
    f"Con **{len(context['names'])} clases**, la parte que entrenamos tiene "
    f"**{(2048 + 1) * len(context['names']):,} parámetros**. "
    "ResNet50 permanece fija y se informa por separado como coste del extractor."
))


## 4. Medir antes de lanzar el trabajo largo

Primero se descargan los pesos, solo si has activado esa opción. Después, `MEDIR_COSTE=True` permite analizar hasta **tres lotes de 32 fotos** para orientar la duración de la extracción completa. Los vectores de esas fotos ya se guardan y se reutilizan: no se tiran.

La estimación depende del ordenador, del disco y de otros programas abiertos. No es una promesa de duración. Mide lectura, comprobación, paso por DINO y guardado; **no incluye el preentrenamiento externo ni el posterior aprendizaje del clasificador**.

El piloto sirve para medir tiempo, no para elegir un modelo según su accuracy. No impone una muestra de train ni un límite oculto de horas. Revisa el tiempo estimado antes de activar la extracción completa.


In [ ]:
if saved is None and DESCARGAR_PESOS:
    receipt = prepare_weights(context, allow_download=True)
    display(Markdown(f"Pesos disponibles. Huella SHA-256: `{receipt['sha256']}`."))
elif saved is not None:
    display(Markdown('Los resultados ya están guardados: para leerlos no se necesitan los pesos.'))
else:
    display(Markdown('Descarga desactivada. No se ha accedido a internet.'))

if saved is None and MEDIR_COSTE:
    extract_features(context, pilot=True)

pilot_path = cost_path(context)
if pilot_path.is_file():
    pilot = json.loads(pilot_path.read_text(encoding='utf-8'))
    display_es(pd.DataFrame([{
        'Fotos del piloto': pilot['images_processed'],
        'Parámetros fijos de ResNet50': pilot['frozen_encoder_parameters'],
        'Segundos medidos': pilot['seconds'],
        'Segundos por foto': pilot['seconds_per_image'],
        'Minutos restantes estimados al medir': pilot['remaining_estimate_seconds'] / 60,
        'Pico RSS del proceso (MiB)': pilot['peak_rss_bytes'] / 2**20,
    }]))
    display(Markdown('El tiempo restante es el estimado **cuando se hizo el piloto**, '
                     'no un contador actualizado. RSS es memoria del proceso completo, no solo de los tensores.'))
else:
    display(Markdown('Aún no hay medición de coste para esta configuración. No se ha iniciado el piloto.'))


## 5. Pasar las fotos por la red una vez y guardar sus resúmenes

Con la parte que mira las fotos fija, cada imagen produce siempre el mismo vector bajo este preprocesamiento. Por eso lo guardamos **en archivos del ordenador**, dentro de `tmp/07_transfer/features`, no en la memoria del kernel.

La caché usa la huella del JPEG, de los pesos y la receta de preparación. Cada lote queda confirmado en disco antes de pasar al siguiente. Si se interrumpe, se conservan los lotes completos. Al reanudar con la opción correspondiente, se procesan solo las fotos que faltan.

**Cambiar una métrica o un texto no vuelve a analizar las fotos.** Si retiramos productos, los vectores de las fotos que siguen sirven mientras no cambien el JPEG, el modelo o la preparación. Cambiar de tamaño, de pesos o de normalización sí requiere vectores distintos.

La extracción utiliza **todas las fotos elegibles de train y validation**. No utiliza fotos de test, que se reservarán para la evaluación final una vez congeladas las decisiones. Esto no cambia el filtrado visual del 02, que sí se aplica a las tres particiones.

> Aunque el encoder no use etiquetas, cambiar sus pesos con validation también sería aprender de validation. Aquí no lo ajustamos con ninguna de esas fotos: solo calculamos sus representaciones.


In [ ]:
if saved is not None:
    display(Markdown('Extracción innecesaria para leer los resultados guardados.'))
elif EXTRAER_FOTOS:
    extraction = extract_features(context, pilot=False)
    if extraction is not None:
        display_es(pd.DataFrame([{
            'Fotos nuevas analizadas': extraction['images_processed'],
            'Minutos de esta extracción': extraction['seconds'] / 60,
            'Segundos por foto': extraction['seconds_per_image'],
        }]))
else:
    display(Markdown('Extracción desactivada. No se han abierto fotos ni ejecutado DINO en esta celda.'))

if saved is None:
    status = feature_status(context)
    display_es(pd.DataFrame([{
        'Fotos necesarias': status['required'],
        'Con vector guardado': status['cached'],
        'Pendientes': status['missing'],
    }]))


## 6. Enseñar a la última capa a decidir la prenda

Esta fase usa los vectores guardados: no vuelve a abrir las imágenes ni a ejecutar ResNet50.

- **Train** sirve para aprender los parámetros de la última capa.
- **Validation** sirve para comprobar cómo va y elegir qué versión guardamos. No actualiza pesos.
- **Test** no participa en este cuaderno.

Usamos las tres semillas del protocolo, 42, 123 y 2026. Una semilla fija la inicialización y el orden del entrenamiento para poder repetirlo. Mostramos las tres ejecuciones; no seleccionamos la semilla con mejor resultado.

Se mantiene Adam, tasa de aprendizaje 0,001, lotes de 32 y un máximo de 30 épocas. Una época recorre todos los ejemplos de train. Paramos tras cinco épocas seguidas sin disminuir la pérdida de validation. Entre las épocas alcanzadas guardamos la de mayor macro-F1 de validation; si empatan, la primera. **Detener y elegir son dos decisiones distintas.**

La pérdida es entropía cruzada sin pesos. Recibe las puntuaciones directamente, antes de convertirlas en probabilidades. No equilibramos clases ni eliminamos fotos válidas. Estas decisiones se registran en `config/transfer_v1.json`.

Cada semilla terminada guarda sus predicciones, historial y pequeña capa final. Si se corta a mitad de una semilla, esa semilla empieza de nuevo; las semillas ya terminadas se reutilizan.


In [ ]:
if saved is not None:
    display(Markdown('Clasificadores completos encontrados: no se vuelven a entrenar.'))
elif ENTRENAR_CLASIFICADOR:
    saved = train_heads(context)
else:
    display(Markdown('Entrenamiento desactivado. Actívalo solo cuando los vectores estén completos '
                     'y quieras realizar el primer entrenamiento con este dataset.'))


## 7. Leer los resultados sin repetir el trabajo

**Accuracy** es la fracción total de aciertos. Puede estar dominada por las clases más frecuentes. **Macro-F1** calcula el F1 por clase y les da el mismo peso, de modo que una clase pequeña también cuenta. Se leen juntas y se revisan los resultados por clase.

Las métricas se recalculan desde las predicciones guardadas. Cambiar esta presentación no cambia la identidad del experimento y no entrena de nuevo. Se comprueba que IDs, grupos y etiquetas coinciden con validation actual.

Los tiempos de entrenamiento de esta tabla corresponden **solo al clasificador**. Hay que informar también el piloto, la extracción y el modelo externo reutilizado: una red fija sigue teniendo coste de cálculo y almacenamiento.


In [ ]:
if saved is None:
    display(Markdown('**Resultados pendientes.** No se muestran cifras inventadas ni se reutiliza '
                     'un experimento de otro dataset como si fuese el actual.'))
else:
    metrics, per_class, matrices = evaluate_saved(
        saved, context['rows']['validation'], context['names'])
    display_es(metrics)
    display_es(metrics[['Accuracy', 'Macro-F1']].agg(['mean', 'std']).rename(
        index={'mean': 'Media de las tres semillas', 'std': 'Desviación entre semillas'}))
    display_es(per_class.groupby('Clase')[['precision', 'recall', 'f1-score', 'support']].mean())
    display(Markdown('Una clase con soporte 0 en validation no queda evaluada ahí. '
                     'El valor 0 usado para métricas indefinidas no es una medición de su rendimiento.'))


### 7.1. Curvas: comprobar qué aprende la última capa

Azul representa train y naranja validation. Si train mejora mucho mientras validation empeora, la última capa puede estar memorizando. Si ambas métricas son bajas, la representación, la capa final o el protocolo pueden ser insuficientes. Una curva sola no identifica la causa.

Mostramos la semilla 42 como ilustración fijada de antemano; las tablas anteriores incluyen todas.


In [ ]:
if saved is not None:
    import io
    import matplotlib.pyplot as plt
    first = next(fit for fit in saved[0]['fits'] if fit['seed'] == 42)
    history = pd.read_csv(saved[1] / first['history'])
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
    for ax, field, title in zip(axes, ['loss', 'macro_f1'], ['Pérdida', 'Macro-F1']):
        ax.plot(history.epoch, history['train_' + field], color=BLUE, label='Train')
        ax.plot(history.epoch, history['validation_' + field], color=ORANGE, label='Validation')
        ax.axvline(first['best_epoch'], color='#667085', linestyle=':', linewidth=1)
        ax.set(xlabel='Época', ylabel=title, title=title)
        ax.legend(fontsize=9)
    fig.tight_layout()
    buffer = io.BytesIO()
    fig.savefig(buffer, format='png', dpi=130, bbox_inches='tight')
    plt.close(fig)
    display(DisplayImage(data=buffer.getvalue(), width=850))


### 7.2. Intervalos de confianza: cuánto cambia la evaluación según los productos

Volvemos a formar conjuntos de validation tomando **grupos de productos completos con reemplazo**, 300 veces. Así no tratamos como independientes fotos relacionadas. Los percentiles 2,5 y 97,5 dan un intervalo orientativo del 95 %.

Lo mostramos para la semilla 42, elegida antes de mirar resultados. Es la incertidumbre de evaluación **condicionada a ese modelo**, no la variación del entrenamiento entre semillas. Además, validation ya ha servido para elegir la época: estos intervalos no sustituyen a una evaluación final independiente en test.


In [ ]:
if saved is not None:
    first = next(fit for fit in saved[0]['fits'] if fit['seed'] == 42)
    prediction = pd.read_csv(saved[1] / first['predictions'],
                             dtype={'article_id': 'string', 'group_id': 'string'})
    validation_rows = context['rows']['validation']
    aligned = prediction.set_index('article_id').loc[validation_rows.article_id]
    intervals = product_bootstrap(
        validation_rows, aligned.predicted_id.to_numpy(), len(context['names']),
        draws=config['bootstrap']['draws'], seed=config['bootstrap']['seed'])
    display_es(intervals)


### 7.3. Confusión: qué prendas se mezclan

Cada fila representa una clase real y cada columna una predicción. La diagonal son aciertos. El color muestra la proporción dentro de cada clase real; las clases grandes no dominan la escala únicamente por tener más fotos. La tabla por clase de la sección 7 aporta sus recuentos.

No usamos estas confusiones para cambiar el filtrado ni para seleccionar fotos de test.


In [ ]:
if saved is not None:
    import io
    import matplotlib.pyplot as plt
    from matplotlib.colors import LinearSegmentedColormap
    matrix = matrices[42]
    totals = matrix.sum(axis=1, keepdims=True)
    normalized = np.divide(matrix, totals, out=np.zeros_like(matrix, dtype=float), where=totals > 0)
    cmap = LinearSegmentedColormap.from_list('etsit_07', ['#FFFFFF', BLUE])
    fig, ax = plt.subplots(figsize=(9, 8))
    image = ax.imshow(normalized, vmin=0, vmax=1, cmap=cmap)
    ticks = range(len(context['names']))
    ax.set_xticks(ticks, context['names'], rotation=90, fontsize=9)
    ax.set_yticks(ticks, context['names'], fontsize=9)
    ax.set(xlabel='Prenda predicha', ylabel='Prenda real',
           title='Validation · proporción de cada clase · semilla 42')
    ax.grid(False)
    fig.colorbar(image, ax=ax, shrink=.75)
    fig.tight_layout()
    buffer = io.BytesIO()
    fig.savefig(buffer, format='png', dpi=130, bbox_inches='tight')
    plt.close(fig)
    display(DisplayImage(data=buffer.getvalue(), width=820))


## 8. Qué procede de clase y qué hemos adaptado

Partimos de la idea del notebook **`pytorch_transfer_learning_colab.ipynb`** facilitado por el profesor: reutilizar una red, dejar fija la parte que describe la imagen y sustituir su capa final. El ejemplo de clase usa ResNet18 con pesos supervisados de ImageNet y CIFAR-10.

| Cambio nuestro | Motivo |
|---|---|
| ResNet50 entrenada con DINO, en lugar de los pesos supervisados del ejemplo | Aplicar un preentrenamiento autosupervisado documentado. DINO es una ampliación externa, no un modelo que atribuyamos al notebook del profesor. |
| Fotos de H&M y particiones del 02 | Mantener la tarea y el contrato de datos del proyecto. |
| Imagen completa a 64 × 64 | Compartir resolución con 04–06, conservando la silueta. Se declara la diferencia frente a la evaluación original de DINO a 224. |
| Vectores guardados por foto | No volver a ejecutar la red fija cada vez que calculamos una métrica o entrenamos la capa final. |
| Tres semillas, macro-F1, métricas por clase e IC por productos | Analizar resultados y variación, sin buscar únicamente el mayor porcentaje de acierto. |
| Descarga, piloto, extracción y entrenamiento separados | Medir el coste antes y evitar operaciones largas por sorpresa. |

También conectamos con la distinción entre congelar y ajustar del notebook **`MLLB_Keras_transfer_learning_v2.ipynb`**. No incorporamos BERT a esta tarea visual.

**Fuentes externas y herramientas:** [DINO y sus checkpoints](https://github.com/facebookresearch/dino), [artículo](https://arxiv.org/abs/2104.14294), [preprocesamiento de evaluación de los autores](https://github.com/facebookresearch/dino/blob/main/eval_linear.py), [arquitectura ResNet50 en torchvision](https://docs.pytorch.org/vision/0.22/models/generated/torchvision.models.resnet50.html). La dependencia torchvision 0.22.1 corresponde al PyTorch 2.7.1 ya fijado en el proyecto: [compatibilidad oficial](https://pytorch.org/get-started/previous-versions/#v2-7-1).


## 9. Qué requisito del profesor cubre y qué no debemos dar por terminado

Este experimento aplica **un modelo externo preentrenado mediante autosupervisión** y estudia **transferencia con la red fija**. Explica qué sería el fine-tuning, pero **no lo presenta como un experimento realizado**. Antes de hacerlo, decidiremos qué parte modificar y mediremos su coste.

La conexión con **zero-shot** se desarrolla en el 11: allí se estudia predecir clases sin entrenar un clasificador local para ellas. Aquí sí entrenamos una capa con etiquetas H&M, así que **este experimento no es zero-shot**. DINO por sí solo no relaciona nombres de prendas con imágenes como CLIP.

El 07 tampoco sustituye las arquitecturas del resto del proyecto, el Transformer del 08 ni el generador autorregresivo del 12. Aplicar este modelo es una parte de los requisitos, **no una garantía de que todo el informe los cumpla**.

En el informe final habrá que abrir con el resumen de máximo tres páginas y describir alcance, método, resultados reales y conclusiones. El objetivo de este cuaderno es comprender y analizar una transferencia sencilla, no reproducir un entrenamiento de gran escala.

### Errores de IA detectados durante el diseño

- **Propuesta de usar BERT para continuar la clasificación de fotos.** Se detectó al comprobar su entrada: BERT recibe texto. Se corrigió manteniendo un modelo visual en el 07.
- **Confundir estar preentrenado con estar preentrenado sin etiquetas.** Se contrastó el origen de los pesos: el ejemplo de ResNet18 de clase usa supervisión; el checkpoint DINO elegido documenta autosupervisión.
- **Dar a entender que cambiar resolución permite aislar el efecto del preentrenamiento.** Se detectó al comparar las entradas. Se acordaron 64 × 64 y se explican las otras diferencias que siguen impidiendo esa atribución causal.

Estos son errores de propuesta y explicación, no resultados experimentales inventados. Conservamos el chat como evidencia del aprendizaje y debemos poder explicar las correcciones con nuestras palabras.


## 10. Cómo ejecutarlo y qué compartir con el equipo

**Para leer resultados ya calculados:** deja las cuatro opciones en `False` y ejecuta el cuaderno. Necesitas el proyecto y la publicación vigente del 02. Las métricas y predicciones completas de `results/metrics/07_transfer/runs` permiten leer el experimento sin fotos, pesos ni vectores locales, siempre que dispongas de los metadatos del dataset correspondientes.

**La primera vez, sin resultados:**

1. En `.venv`, instala las dependencias adicionales solo si faltan. Reinicia el kernel si has instalado paquetes.
2. Activa `DESCARGAR_PESOS=True` y `MEDIR_COSTE=True`. Ejecuta hasta la sección 4 y revisa la estimación. No actives todavía la extracción completa.
3. Si decides continuar, deja las dos anteriores en `False`, activa `EXTRAER_FOTOS=True` y vuelve a ejecutar la celda de configuración de la sección 3 para que el kernel reciba el cambio. Después ejecuta la sección 5. Verás avance por lotes. Los resultados se guardan fuera del kernel.
4. Cuando estén todos los vectores, deja `EXTRAER_FOTOS=False`, activa `ENTRENAR_CLASIFICADOR=True` y vuelve a ejecutar la celda de configuración de la sección 3. Después ejecuta desde la sección 6. No vuelve a pasar las fotos por ResNet50.
5. Al terminar sin errores, deja **todas las opciones en False** y guarda el cuaderno con **Ctrl+S**.

No hace falta volver a ejecutar 02 ni 03 por el simple hecho de repetir 07. Si cambia la selección de datos, el cuaderno exige que 02 y 03 correspondan a ella.

**Qué guarda cada cosa:**

| Carpeta | Contenido | Para qué sirve |
|---|---|---|
| `tmp/07_transfer/weights` | Pesos originales y recibo con su huella | Descargar una sola vez y comprobar que no han cambiado. |
| `tmp/07_transfer/features` | Vectores por huella de foto y receta | Reanudar extracción o entrenar otra capa sin analizar de nuevo las mismas fotos. Es local y no se sube automáticamente a Git. |
| `results/metrics/07_transfer/cost` | Medición piloto y extracción | Documentar coste local por separado del clasificador. |
| `results/metrics/07_transfer/runs` | Predicciones, historial, capa final y configuración | Compartir resultados y recalcular métricas sin entrenar. |

Si alguien quiere **entrenar otro clasificador**, necesitará los vectores compatibles compartidos junto con el recibo, o extraerlos una vez. Si solo quiere leer resultados o cambiar las métricas calculadas desde las predicciones, no los necesita. Cambiar la propia red o preparar las imágenes de otra forma sí exige nuevas representaciones.

### Antes de escribir conclusiones

Tras la ejecución revisaremos si aprende, dónde falla y qué coste tiene. Solo compararemos resultados de 04–07 que tengan la misma huella de dataset. Informaremos por separado el coste de extracción, entrenamiento e inferencia de la pequeña capa: el tiempo de esa capa por sí solo no es la latencia completa de clasificar una foto nueva.

**Ahora no adelantamos una conclusión de rendimiento ni damos por realizado el fine-tuning.**
